In [ ]:
import copy
import os
import sys

import numpy as np
import pandas as pd
import torch

# --- Environment detection ---
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    REPO_DIR = "harxhar"
    if not os.path.isdir(REPO_DIR):
        os.system("git clone https://github.com/jamesdchen/harxhar.git " + REPO_DIR)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)
    os.system("pip install -q torch transformers numpy pandas scikit-learn tqdm pyarrow numba")
else:
    sys.path.insert(0, os.path.abspath(".."))

from src.backtest.gpu_engine_scaling import run_scaling_experiment
from src.config import DL_CONFIG
from src.data import load_and_prep_data_strided

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

# --- Experiment Configuration ---
SCALE_MULTIPLIERS = [0, 1, 2, 5, 10, 50]
N_REPEATS = 3
BLOCK_SIZE = 48  # one trading day of 30-min bars
TRAIN_FRAC = 0.8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if IN_COLAB:
    # On Colab: data may come from Drive or upload
    # Uncomment to mount Google Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # DATA_PATH = '/content/drive/MyDrive/harxhar_data/all30min'
    DATA_PATH = "all30min"
    RESULTS_DIR = "results_scaling_laws"
else:
    DATA_PATH = "all30min/"
    RESULTS_DIR = "../results_scaling_laws"

CONFIG = copy.deepcopy(DL_CONFIG)
CONFIG["data_path"] = DATA_PATH

LOADING_HPARAMS = {
    "exog_cols": "none",
    "use_transform_exog": False,
    "use_diurnal": True,
    "allow_missing": False,
    "use_winsor": False,
}

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Device: {DEVICE}")
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Results dir: {os.path.abspath(RESULTS_DIR)}")

In [ ]:
# Load data using the codebase pipeline (same as patchts_backtest.ipynb)
X_np, y_np, dates, baselines, features = load_and_prep_data_strided(
    LOADING_HPARAMS, CONFIG["data_path"], lag=CONFIG["model"]["context_len"]
)
print(f"Data: {X_np.shape[0]} samples, {X_np.shape[1]} features")
print(f"Target range: [{y_np.min():.4f}, {y_np.max():.4f}]")

In [ ]:
# Strict chronological split: 80% train, 20% test
split_idx = int(len(X_np) * TRAIN_FRAC)

X_train, y_train = X_np[:split_idx], y_np[:split_idx]
X_test, y_test = X_np[split_idx:], y_np[split_idx:]
baselines_test = baselines[split_idx:]
dates_test = dates.iloc[split_idx:]

print(f"Train: {len(X_train)} samples ({TRAIN_FRAC:.0%})")
print(f"Test:  {len(X_test)} samples ({1-TRAIN_FRAC:.0%})")
print(f"Test date range: {dates_test.iloc[0]} → {dates_test.iloc[-1]}")

In [ ]:
# Run scaling experiments — results saved per-multiplier for fault tolerance
all_results = []
csv_path = os.path.join(RESULTS_DIR, "scaling_results.csv")

# Resume from previous partial run if the CSV exists (useful for Colab disconnects)
if os.path.exists(csv_path):
    prev_df = pd.read_csv(csv_path)
    done_keys = set(zip(prev_df["multiplier"], prev_df["repeat"]))
    all_results = prev_df.to_dict("records")
    print(f"Resuming: {len(done_keys)} runs already completed")
else:
    done_keys = set()

for mult in SCALE_MULTIPLIERS:
    for rep in range(N_REPEATS):
        if (mult, rep) in done_keys:
            print(f"Skipping multiplier={mult}, repeat={rep} (already done)")
            continue

        seed = mult * 1000 + rep
        print(f"\n{'='*60}")
        print(f"Multiplier={mult}, Repeat={rep}, Seed={seed}")
        print(f"{'='*60}")

        result = run_scaling_experiment(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            baselines_test=baselines_test,
            model_config=CONFIG["model"],
            train_config=CONFIG["train"],
            multiplier=mult,
            block_size=BLOCK_SIZE,
            seed=seed,
            device=DEVICE,
        )
        result["repeat"] = rep
        all_results.append(result)

        # Save incrementally for fault tolerance
        pd.DataFrame(all_results).drop(columns=["epoch_losses"], errors="ignore").to_csv(
            csv_path, index=False
        )
        print(f"  → QLIKE={result['qlike']:.6f}, n_windows={result['n_train_windows']}")

print(f"\nAll results saved to {csv_path}")

In [ ]:
# Quick summary
df = pd.read_csv(csv_path)
summary = df.groupby("multiplier").agg(
    qlike_mean=("qlike", "mean"),
    qlike_std=("qlike", "std"),
    mse_mean=("mse", "mean"),
    mae_mean=("mae", "mean"),
    n_windows=("n_train_windows", "first"),
).reset_index()
print(summary.to_string(index=False))

# Download results on Colab
if IN_COLAB:
    try:
        from google.colab import files
        files.download(csv_path)
    except Exception:
        pass